# **Table of Contents**
* [Your Notebook](#your-notebook)
  * [Table of Contents](#table-of-contents)
* [TOC Generator](#toc-generator)


imports

In [1]:
import pandas as pd

## Clean Poverty Data

In [2]:
df_poverty = pd.read_csv('../Curtis/data/raw/acs_2024_jefferson_poverty_by_tract_s1701.csv')

df_poverty.columns

Index(['GEO_ID', 'NAME', 'S1701_C01_001E', 'S1701_C01_001M', 'S1701_C01_002E',
       'S1701_C01_002M', 'S1701_C01_003E', 'S1701_C01_003M', 'S1701_C01_004E',
       'S1701_C01_004M',
       ...
       'S1701_C03_058M', 'S1701_C03_059E', 'S1701_C03_059M', 'S1701_C03_060E',
       'S1701_C03_060M', 'S1701_C03_061E', 'S1701_C03_061M', 'S1701_C03_062E',
       'S1701_C03_062M', 'Unnamed: 374'],
      dtype='object', length=375)

In [3]:
df_poverty = df_poverty[["GEO_ID", "NAME", "S1701_C01_001E", "S1701_C02_001E", "S1701_C03_001E"]]

# The first row labels are redundant
df_poverty.drop(index = 0, inplace = True)

# Rename to useful column names
df_poverty = df_poverty.rename(columns={
    "NAME" : "TRACT",
    "S1701_C01_001E" : "POPULATION",
    "S1701_C02_001E" : "BELOW_POVERTY",
    "S1701_C03_001E" : "POVERTY_RATE"
    })

# Keep only tract # in string
df_poverty['TRACT'] = df_poverty['TRACT'].str.extract(r'Census Tract ([\d.]+)')

# Treat Tract # as a float for merging later (hold if needed)
# df_poverty['TRACT'] = df_poverty['TRACT'].astype(float)

df_poverty.head()

,GEO_ID,TRACT,POPULATION,BELOW_POVERTY,POVERTY_RATE
1,1400000US21111000201,2.01,1163,398,34.2
2,1400000US21111000202,2.02,706,370,52.4
3,1400000US21111000300,3,2155,585,27.1
4,1400000US21111000400,4,4644,955,20.6
5,1400000US21111000600,6,1252,328,26.2


In [4]:
# Output clean file
df_poverty.to_csv("../Curtis/data/processed/poverty_rate_clean.csv", index = False)

## Clean Income Data

In [5]:
df_income = pd.read_csv('../Curtis/data/raw/acs_2024_jefferson_household_income_by_tract_b19013.csv')

df_income.head()

,Label (Grouping),Median household income in the past 12 months (in 2024 inflation-adjusted dollars)
0,Census Tract 2.01; Jefferson County; Kentucky,NaN
1,Estimate,"35,825"
2,Margin of Error,"±11,473"
3,Census Tract 2.02; Jefferson County; Kentucky,NaN
4,Estimate,"17,039"


In [6]:
# Rename columns
df_income = df_income.rename(columns={
    "Label (Grouping)" : "TRACT",
    "Median household income in the past 12 months (in 2024 inflation-adjusted dollars)" : "ESTIMATED_MEDIAN_HOUSEHOLD_INCOME"
    })

df_income.head()

,TRACT,ESTIMATED_MEDIAN_HOUSEHOLD_INCOME
0,Census Tract 2.01; Jefferson County; Kentucky,NaN
1,Estimate,"35,825"
2,Margin of Error,"±11,473"
3,Census Tract 2.02; Jefferson County; Kentucky,NaN
4,Estimate,"17,039"


In [7]:
# Shift values up one row so Estimate will be on the same row as tract
df_income['ESTIMATED_MEDIAN_HOUSEHOLD_INCOME'] = df_income['ESTIMATED_MEDIAN_HOUSEHOLD_INCOME'].shift(-1)

df_income.head()

,TRACT,ESTIMATED_MEDIAN_HOUSEHOLD_INCOME
0,Census Tract 2.01; Jefferson County; Kentucky,"35,825"
1,Estimate,"±11,473"
2,Margin of Error,NaN
3,Census Tract 2.02; Jefferson County; Kentucky,"17,039"
4,Estimate,"±6,250"


In [8]:
# Simplify the layout to be "Tract #: Estimate value" and delete Margin of Error rows
df_income = df_income.drop(df_income[df_income['TRACT'].str.contains('Estimate|Margin of Error', na=False)].index)

# Keep only tract # in string
df_income['TRACT'] = df_income['TRACT'].str.extract(r'Census Tract ([\d.]+)')

# Treat Tract # as a float for merging later (hold if needed)
# df_income['TRACT'] = df_income['TRACT'].astype(float)

df_income.head()

,TRACT,ESTIMATED_MEDIAN_HOUSEHOLD_INCOME
0,2.01,"35,825"
3,2.02,"17,039"
6,3,"34,764"
9,4,"41,713"
12,6,"30,734"


In [9]:
# Output clean file
df_income.to_csv("../Curtis/data/processed/income_clean.csv", index = True)

## Clean Housing Age Data

In [10]:
df_housing_age = pd.read_csv('../Curtis/data/raw/acs_2024_jefferson_housing_age_by_tract_b25034.csv')

df_housing_age.head()

,Label (Grouping),Census Tract 2.01; Jefferson County; Kentucky!!Estimate,Census Tract 2.01; Jefferson County; Kentucky!!Margin of Error,Census Tract 2.02; Jefferson County; Kentucky!!Estimate,Census Tract 2.02; Jefferson County; Kentucky!!Margin of Error,Census Tract 3; Jefferson County; Kentucky!!Estimate,Census Tract 3; Jefferson County; Kentucky!!Margin of Error,Census Tract 4; Jefferson County; Kentucky!!Estimate,Census Tract 4; Jefferson County; Kentucky!!Margin of Error,Census Tract 6; Jefferson County; Kentucky!!Estimate,...,Census Tract 127.03; Jefferson County; Kentucky!!Estimate,Census Tract 127.03; Jefferson County; Kentucky!!Margin of Error,Census Tract 128.01; Jefferson County; Kentucky!!Estimate,Census Tract 128.01; Jefferson County; Kentucky!!Margin of Error,Census Tract 128.02; Jefferson County; Kentucky!!Estimate,Census Tract 128.02; Jefferson County; Kentucky!!Margin of Error,Census Tract 131; Jefferson County; Kentucky!!Estimate,Census Tract 131; Jefferson County; Kentucky!!Margin of Error,Census Tract 9801; Jefferson County; Kentucky!!Estimate,Census Tract 9801; Jefferson County; Kentucky!!Margin of Error
0,Total:,777,±125,519,±118,"1,085",±149,"2,045",±230,866,...,"2,553",±264,"1,366",±225,"1,262",±252,503,±55,0,±13
1,Built 2020 or later,8,±14,0,±13,0,±13,0,±13,36,...,76,±81,0,±13,0,±13,2,±2,0,±13
2,Built 2010 to 2019,0,±13,0,±13,53,±79,0,±13,7,...,1,±10,64,±80,31,±39,4,±5,0,±13
3,Built 2000 to 2009,15,±22,14,±17,0,±13,42,±50,159,...,615,±239,0,±13,0,±13,5,±4,0,±13
4,Built 1990 to 1999,47,±69,39,±40,42,±20,0,±13,118,...,448,±241,17,±24,22,±35,4,±4,0,±13


In [11]:
# Drop all Estimate columns
cols_to_drop_housing_age = df_housing_age.columns[df_housing_age.columns.str.contains('Margin of Error')]
df_housing_age = df_housing_age.drop(columns=cols_to_drop_housing_age)

df_housing_age.head()

,Label (Grouping),Census Tract 2.01; Jefferson County; Kentucky!!Estimate,Census Tract 2.02; Jefferson County; Kentucky!!Estimate,Census Tract 3; Jefferson County; Kentucky!!Estimate,Census Tract 4; Jefferson County; Kentucky!!Estimate,Census Tract 6; Jefferson County; Kentucky!!Estimate,Census Tract 7; Jefferson County; Kentucky!!Estimate,Census Tract 8; Jefferson County; Kentucky!!Estimate,Census Tract 9; Jefferson County; Kentucky!!Estimate,Census Tract 10; Jefferson County; Kentucky!!Estimate,...,Census Tract 126.04; Jefferson County; Kentucky!!Estimate,Census Tract 126.05; Jefferson County; Kentucky!!Estimate,Census Tract 126.06; Jefferson County; Kentucky!!Estimate,Census Tract 127.01; Jefferson County; Kentucky!!Estimate,Census Tract 127.02; Jefferson County; Kentucky!!Estimate,Census Tract 127.03; Jefferson County; Kentucky!!Estimate,Census Tract 128.01; Jefferson County; Kentucky!!Estimate,Census Tract 128.02; Jefferson County; Kentucky!!Estimate,Census Tract 131; Jefferson County; Kentucky!!Estimate,Census Tract 9801; Jefferson County; Kentucky!!Estimate
0,Total:,777,519,"1,085","2,045",866,"1,331",975,999,"1,393",...,"2,351","1,634","1,487","2,025",852,"2,553","1,366","1,262",503,0
1,Built 2020 or later,8,0,0,0,36,0,0,0,0,...,29,0,0,29,0,76,0,0,2,0
2,Built 2010 to 2019,0,0,53,0,7,0,0,5,0,...,21,11,0,459,25,1,64,31,4,0
3,Built 2000 to 2009,15,14,0,42,159,29,35,4,57,...,372,15,70,225,68,615,0,0,5,0
4,Built 1990 to 1999,47,39,42,0,118,15,21,57,0,...,45,12,114,38,0,448,17,22,4,0


In [12]:
# Swap columns and rows to match other data sets
df_housing_age = df_housing_age.transpose()

# Set the first row as the header
df_housing_age = df_housing_age.rename(columns=df_housing_age.iloc[0]).drop(df_housing_age.index[0])

df_housing_age.head()

,Total:,Built 2020 or later,Built 2010 to 2019,Built 2000 to 2009,Built 1990 to 1999,Built 1980 to 1989,Built 1970 to 1979,Built 1960 to 1969,Built 1950 to 1959,Built 1940 to 1949,Built 1939 or earlier
Census Tract 2.01; Jefferson County; Kentucky!!Estimate,777,8,0,15,47,37,54,20,89,61,446
Census Tract 2.02; Jefferson County; Kentucky!!Estimate,519,0,0,14,39,0,0,13,8,103,342
Census Tract 3; Jefferson County; Kentucky!!Estimate,"1,085",0,53,0,42,0,57,36,149,116,632
Census Tract 4; Jefferson County; Kentucky!!Estimate,"2,045",0,0,42,0,65,178,40,428,188,"1,104"
Census Tract 6; Jefferson County; Kentucky!!Estimate,866,36,7,159,118,102,9,65,55,106,209


In [13]:
# Remove the spaces before "Built" from all column names
df_housing_age.columns = df_housing_age.columns.str.strip()

# Remove the colon from "Total:"
df_housing_age.rename(columns={'Total:': 'TOTAL'}, inplace=True)

# Create index column
df_housing_age = df_housing_age.reset_index() 

# Now we can label first column
df_housing_age.rename(columns={df_housing_age.columns[0]: 'TRACT'}, inplace=True)

# Keep only tract # in string
df_housing_age['TRACT'] = df_housing_age['TRACT'].str.extract(r'Census Tract ([\d.]+)')

df_housing_age.head()

,TRACT,TOTAL,Built 2020 or later,Built 2010 to 2019,Built 2000 to 2009,Built 1990 to 1999,Built 1980 to 1989,Built 1970 to 1979,Built 1960 to 1969,Built 1950 to 1959,Built 1940 to 1949,Built 1939 or earlier
0,2.01,777,8,0,15,47,37,54,20,89,61,446
1,2.02,519,0,0,14,39,0,0,13,8,103,342
2,3,"1,085",0,53,0,42,0,57,36,149,116,632
3,4,"2,045",0,0,42,0,65,178,40,428,188,"1,104"
4,6,866,36,7,159,118,102,9,65,55,106,209


In [14]:
# Output clean file
df_housing_age.to_csv("../Curtis/data/processed/housing_age_clean.csv", index = True)

## Clean Median Gross Rent

In [15]:
df_rent = pd.read_csv('../Curtis/data/raw/acs_2024_jefferson_median_gross_rent_by_tract_b25064.csv')

df_rent.head()

,Label (Grouping),Census Tract 2.01; Jefferson County; Kentucky!!Estimate,Census Tract 2.01; Jefferson County; Kentucky!!Margin of Error,Census Tract 2.02; Jefferson County; Kentucky!!Estimate,Census Tract 2.02; Jefferson County; Kentucky!!Margin of Error,Census Tract 3; Jefferson County; Kentucky!!Estimate,Census Tract 3; Jefferson County; Kentucky!!Margin of Error,Census Tract 4; Jefferson County; Kentucky!!Estimate,Census Tract 4; Jefferson County; Kentucky!!Margin of Error,Census Tract 6; Jefferson County; Kentucky!!Estimate,...,Census Tract 127.03; Jefferson County; Kentucky!!Estimate,Census Tract 127.03; Jefferson County; Kentucky!!Margin of Error,Census Tract 128.01; Jefferson County; Kentucky!!Estimate,Census Tract 128.01; Jefferson County; Kentucky!!Margin of Error,Census Tract 128.02; Jefferson County; Kentucky!!Estimate,Census Tract 128.02; Jefferson County; Kentucky!!Margin of Error,Census Tract 131; Jefferson County; Kentucky!!Estimate,Census Tract 131; Jefferson County; Kentucky!!Margin of Error,Census Tract 9801; Jefferson County; Kentucky!!Estimate,Census Tract 9801; Jefferson County; Kentucky!!Margin of Error
0,Median gross rent,"1,141",±264,"1,288",±58,"1,234",±148,"1,012",±149,752,...,"1,445",±396,678,±292,740,±483,"1,411",±528,-,**


In [16]:
# Drop all Estimate columns
cols_to_drop_rent = df_rent.columns[df_rent.columns.str.contains('Margin of Error')]
df_rent = df_rent.drop(columns=cols_to_drop_rent)

# Swap columns and rows to match other data sets
df_rent = df_rent.transpose()

df_rent.head()

,0
Label (Grouping),Median gross rent
Census Tract 2.01; Jefferson County; Kentucky!!Estimate,"1,141"
Census Tract 2.02; Jefferson County; Kentucky!!Estimate,"1,288"
Census Tract 3; Jefferson County; Kentucky!!Estimate,"1,234"
Census Tract 4; Jefferson County; Kentucky!!Estimate,"1,012"


In [17]:
df_rent = df_rent.reset_index()

# Set the first row as the header
df_rent = df_rent.rename(columns=df_rent.iloc[0]).drop(df_rent.index[0])

df_rent.rename(columns={'Median gross rent' : 'MEDIAN_GROSS_RENT'}, inplace = True)

df_rent.head()

,Label (Grouping),MEDIAN_GROSS_RENT
1,Census Tract 2.01; Jefferson County; Kentucky!...,"1,141"
2,Census Tract 2.02; Jefferson County; Kentucky!...,"1,288"
3,Census Tract 3; Jefferson County; Kentucky!!Es...,"1,234"
4,Census Tract 4; Jefferson County; Kentucky!!Es...,"1,012"
5,Census Tract 6; Jefferson County; Kentucky!!Es...,752


In [18]:
# Now we can label first column
df_rent.rename(columns={df_rent.columns[0]: 'TRACT'}, inplace=True)

# Keep only tract # in string
df_rent['TRACT'] = df_rent['TRACT'].str.extract(r'Census Tract ([\d.]+)')

df_rent.head()


,TRACT,MEDIAN_GROSS_RENT
1,2.01,"1,141"
2,2.02,"1,288"
3,3,"1,234"
4,4,"1,012"
5,6,752


In [19]:
# Output clean file
df_rent.to_csv("../Curtis/data/processed/rent_clean.csv", index = True)

## Clean Home Value

In [22]:
df_home_value = pd.read_csv('../Curtis/data/raw/acs_2024_jefferson_median_home_value_by_tract_b25077.csv')

df_home_value.head()

,Label (Grouping),Census Tract 2.01; Jefferson County; Kentucky!!Estimate,Census Tract 2.01; Jefferson County; Kentucky!!Margin of Error,Census Tract 2.02; Jefferson County; Kentucky!!Estimate,Census Tract 2.02; Jefferson County; Kentucky!!Margin of Error,Census Tract 3; Jefferson County; Kentucky!!Estimate,Census Tract 3; Jefferson County; Kentucky!!Margin of Error,Census Tract 4; Jefferson County; Kentucky!!Estimate,Census Tract 4; Jefferson County; Kentucky!!Margin of Error,Census Tract 6; Jefferson County; Kentucky!!Estimate,...,Census Tract 127.03; Jefferson County; Kentucky!!Estimate,Census Tract 127.03; Jefferson County; Kentucky!!Margin of Error,Census Tract 128.01; Jefferson County; Kentucky!!Estimate,Census Tract 128.01; Jefferson County; Kentucky!!Margin of Error,Census Tract 128.02; Jefferson County; Kentucky!!Estimate,Census Tract 128.02; Jefferson County; Kentucky!!Margin of Error,Census Tract 131; Jefferson County; Kentucky!!Estimate,Census Tract 131; Jefferson County; Kentucky!!Margin of Error,Census Tract 9801; Jefferson County; Kentucky!!Estimate,Census Tract 9801; Jefferson County; Kentucky!!Margin of Error
0,Median value (dollars),"44,800","±14,210","82,200","±49,945","64,300","±12,500","105,700","±11,949","62,700",...,"172,900","±30,773","119,400","±23,729","102,600","±8,615","369,600","±9,696",-,**


In [23]:
# Drop all Estimate columns
cols_to_drop_rent = df_home_value.columns[df_home_value.columns.str.contains('Margin of Error')]
df_home_value = df_home_value.drop(columns=cols_to_drop_rent)

# Swap columns and rows to match other data sets
df_home_value = df_home_value.transpose()

df_home_value.head()

,0
Label (Grouping),Median value (dollars)
Census Tract 2.01; Jefferson County; Kentucky!!Estimate,"44,800"
Census Tract 2.02; Jefferson County; Kentucky!!Estimate,"82,200"
Census Tract 3; Jefferson County; Kentucky!!Estimate,"64,300"
Census Tract 4; Jefferson County; Kentucky!!Estimate,"105,700"


In [24]:
df_home_value = df_home_value.reset_index()

# Set the first row as the header
df_home_value = df_home_value.rename(columns=df_home_value.iloc[0]).drop(df_home_value.index[0])

df_home_value.rename(columns={'Median gross rent' : 'MEDIAN_GROSS_RENT'}, inplace = True)

df_home_value.head()

,Label (Grouping),Median value (dollars)
1,Census Tract 2.01; Jefferson County; Kentucky!...,"44,800"
2,Census Tract 2.02; Jefferson County; Kentucky!...,"82,200"
3,Census Tract 3; Jefferson County; Kentucky!!Es...,"64,300"
4,Census Tract 4; Jefferson County; Kentucky!!Es...,"105,700"
5,Census Tract 6; Jefferson County; Kentucky!!Es...,"62,700"


In [25]:
# Now we can label first column
df_home_value.rename(columns={df_home_value.columns[0]: 'TRACT'}, inplace=True)

# Keep only tract # in string
df_home_value['TRACT'] = df_home_value['TRACT'].str.extract(r'Census Tract ([\d.]+)')

df_home_value.head()

,TRACT,Median value (dollars)
1,2.01,"44,800"
2,2.02,"82,200"
3,3,"64,300"
4,4,"105,700"
5,6,"62,700"


In [26]:
# Output clean file
df_rent.to_csv("../Curtis/data/processed/home_value_clean.csv", index = True)

## Clean Rent Burden

In [29]:
df_rent_burden = pd.read_csv('../Curtis/data/raw/acs_2024_jefferson_rent_burden_by_tract_b25070.csv')

df_rent_burden.head(11)

,Label (Grouping),Census Tract 2.01; Jefferson County; Kentucky!!Estimate,Census Tract 2.01; Jefferson County; Kentucky!!Margin of Error,Census Tract 2.02; Jefferson County; Kentucky!!Estimate,Census Tract 2.02; Jefferson County; Kentucky!!Margin of Error,Census Tract 3; Jefferson County; Kentucky!!Estimate,Census Tract 3; Jefferson County; Kentucky!!Margin of Error,Census Tract 4; Jefferson County; Kentucky!!Estimate,Census Tract 4; Jefferson County; Kentucky!!Margin of Error,Census Tract 6; Jefferson County; Kentucky!!Estimate,...,Census Tract 127.03; Jefferson County; Kentucky!!Estimate,Census Tract 127.03; Jefferson County; Kentucky!!Margin of Error,Census Tract 128.01; Jefferson County; Kentucky!!Estimate,Census Tract 128.01; Jefferson County; Kentucky!!Margin of Error,Census Tract 128.02; Jefferson County; Kentucky!!Estimate,Census Tract 128.02; Jefferson County; Kentucky!!Margin of Error,Census Tract 131; Jefferson County; Kentucky!!Estimate,Census Tract 131; Jefferson County; Kentucky!!Margin of Error,Census Tract 9801; Jefferson County; Kentucky!!Estimate,Census Tract 9801; Jefferson County; Kentucky!!Margin of Error
0,Total:,277,±82,188,±52,438,±139,880,±185,461,...,570,±205,334,±124,448,±194,49,±16,0,±13
1,Less than 10.0 percent,0,±13,0,±13,11,±18,55,±93,10,...,57,±65,7,±12,12,±21,0,±13,0,±13
2,10.0 to 14.9 percent,8,±14,0,±13,0,±13,18,±21,78,...,0,±18,44,±69,87,±67,4,±5,0,±13
3,15.0 to 19.9 percent,13,±19,0,±13,51,±60,100,±109,23,...,196,±169,27,±30,0,±13,23,±16,0,±13
4,20.0 to 24.9 percent,75,±54,13,±22,77,±74,12,±19,33,...,63,±64,0,±13,43,±50,3,±4,0,±13
5,25.0 to 29.9 percent,47,±69,17,±20,24,±39,70,±56,42,...,17,±34,12,±19,113,±158,0,±13,0,±13
6,30.0 to 34.9 percent,0,±13,0,±13,19,±21,218,±161,42,...,74,±106,39,±61,15,±24,0,±13,0,±13
7,35.0 to 39.9 percent,0,±13,0,±13,93,±97,0,±13,7,...,0,±18,14,±23,0,±13,3,±4,0,±13
8,40.0 to 49.9 percent,37,±46,15,±23,54,±29,156,±110,56,...,0,±18,74,±80,14,±24,3,±5,0,±13
9,50.0 percent or more,86,±68,122,±30,81,±67,240,±137,163,...,107,±89,92,±70,123,±98,9,±6,0,±13


In [30]:
# Drop all Estimate columns
cols_to_drop_rent_burden = df_rent_burden.columns[df_rent_burden.columns.str.contains('Margin of Error')]
df_rent_burden = df_rent_burden.drop(columns=cols_to_drop_rent_burden)

df_rent_burden.head()

,Label (Grouping),Census Tract 2.01; Jefferson County; Kentucky!!Estimate,Census Tract 2.02; Jefferson County; Kentucky!!Estimate,Census Tract 3; Jefferson County; Kentucky!!Estimate,Census Tract 4; Jefferson County; Kentucky!!Estimate,Census Tract 6; Jefferson County; Kentucky!!Estimate,Census Tract 7; Jefferson County; Kentucky!!Estimate,Census Tract 8; Jefferson County; Kentucky!!Estimate,Census Tract 9; Jefferson County; Kentucky!!Estimate,Census Tract 10; Jefferson County; Kentucky!!Estimate,...,Census Tract 126.04; Jefferson County; Kentucky!!Estimate,Census Tract 126.05; Jefferson County; Kentucky!!Estimate,Census Tract 126.06; Jefferson County; Kentucky!!Estimate,Census Tract 127.01; Jefferson County; Kentucky!!Estimate,Census Tract 127.02; Jefferson County; Kentucky!!Estimate,Census Tract 127.03; Jefferson County; Kentucky!!Estimate,Census Tract 128.01; Jefferson County; Kentucky!!Estimate,Census Tract 128.02; Jefferson County; Kentucky!!Estimate,Census Tract 131; Jefferson County; Kentucky!!Estimate,Census Tract 9801; Jefferson County; Kentucky!!Estimate
0,Total:,277,188,438,880,461,856,261,531,530,...,754,625,398,"1,097",114,570,334,448,49,0
1,Less than 10.0 percent,0,0,11,55,10,111,0,41,0,...,0,0,62,0,0,57,7,12,0,0
2,10.0 to 14.9 percent,8,0,0,18,78,0,28,27,60,...,74,118,1,91,0,0,44,87,4,0
3,15.0 to 19.9 percent,13,0,51,100,23,0,14,0,58,...,218,93,64,14,4,196,27,0,23,0
4,20.0 to 24.9 percent,75,13,77,12,33,100,13,60,67,...,69,22,101,208,6,63,0,43,3,0


In [31]:
# Swap columns and rows to match other data sets
df_rent_burden = df_rent_burden.transpose()

# Set the first row as the header
df_rent_burden = df_rent_burden.rename(columns=df_rent_burden.iloc[0]).drop(df_rent_burden.index[0])

df_rent_burden.head()

,Total:,Less than 10.0 percent,10.0 to 14.9 percent,15.0 to 19.9 percent,20.0 to 24.9 percent,25.0 to 29.9 percent,30.0 to 34.9 percent,35.0 to 39.9 percent,40.0 to 49.9 percent,50.0 percent or more,Not computed
Census Tract 2.01; Jefferson County; Kentucky!!Estimate,277,0,8,13,75,47,0,0,37,86,11
Census Tract 2.02; Jefferson County; Kentucky!!Estimate,188,0,0,0,13,17,0,0,15,122,21
Census Tract 3; Jefferson County; Kentucky!!Estimate,438,11,0,51,77,24,19,93,54,81,28
Census Tract 4; Jefferson County; Kentucky!!Estimate,880,55,18,100,12,70,218,0,156,240,11
Census Tract 6; Jefferson County; Kentucky!!Estimate,461,10,78,23,33,42,42,7,56,163,7


In [32]:
# Remove the white spaces from all column names
df_rent_burden.columns = df_rent_burden.columns.str.strip()

# Remove the colon from "Total:"
df_rent_burden.rename(columns={'Total:': 'TOTAL'}, inplace=True)

# Create index column
df_rent_burden = df_rent_burden.reset_index() 

# Now we can label first column
df_rent_burden.rename(columns={df_rent_burden.columns[0]: 'TRACT'}, inplace=True)

# Keep only tract # in string
df_rent_burden['TRACT'] = df_rent_burden['TRACT'].str.extract(r'Census Tract ([\d.]+)')

df_rent_burden.head()

,TRACT,TOTAL,Less than 10.0 percent,10.0 to 14.9 percent,15.0 to 19.9 percent,20.0 to 24.9 percent,25.0 to 29.9 percent,30.0 to 34.9 percent,35.0 to 39.9 percent,40.0 to 49.9 percent,50.0 percent or more,Not computed
0,2.01,277,0,8,13,75,47,0,0,37,86,11
1,2.02,188,0,0,0,13,17,0,0,15,122,21
2,3,438,11,0,51,77,24,19,93,54,81,28
3,4,880,55,18,100,12,70,218,0,156,240,11
4,6,461,10,78,23,33,42,42,7,56,163,7


In [33]:
# Output clean file
df_rent_burden.to_csv("../Curtis/data/processed/rent_burden_clean.csv", index = True)

# TOC Generator 

Do not delete code. It will read your markdown and generate a TOC we can put in the notebook for easy navigation. 

In [ ]:
import json
import os


def generate_toc_from_notebook(notebook_path):
    """
    Parses a local .ipynb file and generates Markdown for a Table of Contents.
    """
    if not os.path.isfile(notebook_path):
        print(f"❌ Error: File not found at '{notebook_path}'")
        return

    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook = json.load(f)

    toc_markdown = "# **Table of Contents**\n"
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') == 'markdown':
            for line in cell.get('source', []):
                if line.strip().startswith('#'):
                    level = line.count('#')
                    title = line.strip('#').strip()
                    link = title.lower().replace(' ', '-').strip('-.()')
                    indent = '  ' * (level - 1)
                    toc_markdown += f"{indent}* [{title}](#{link})\n"

    print("\n--- ✅ Copy the Markdown below and paste"
          "it into a new markdown cell ---\n")
    print(toc_markdown)


if __name__ == "__main__":
    # Example usage
    notebook_path = 'curtis.ipynb'  # Replace with your notebook path
    generate_toc_from_notebook(notebook_path)


--- ✅ Copy the Markdown below and pasteit into a new markdown cell ---

# **Table of Contents**
* [**Table of Contents**](#**table-of-contents**)
  * [Clean Poverty Data](#clean-poverty-data)
  * [Clean Income Data](#clean-income-data)
  * [Clean Housing Age Data](#clean-housing-age-data)
  * [Clean Median Gross Rent](#clean-median-gross-rent)
* [TOC Generator](#toc-generator)

